# Autoresearch on Colab

Based on [karpathy/autoresearch](https://github.com/karpathy/autoresearch)

**Runtime → Change runtime type → T4 GPU → Run All**

Features:
- T4/Pre-Ampere auto-patch (Flash Attention → SDPA fallback)
- Google Drive auto-save (results survive disconnect)
- Keep-alive (prevents idle timeout)
- OOM recovery with auto batch-size reduction
- Resume from last completed round after reconnect

In [ ]:
# ====== Cell 1: Keep-Alive + GPU Check ======
# JavaScript keep-alive to prevent Colab idle disconnect
from IPython.display import display, Javascript
display(Javascript('''
function KeepAlive() {
  // Click the connect button if it appears
  var buttons = document.querySelectorAll("colab-connect-button");
  if (buttons.length > 0) { buttons[0].click(); }
  // Also simulate activity
  console.log("[keep-alive] " + new Date().toISOString());
}
// Run every 60 seconds
var _keepAliveInterval = setInterval(KeepAlive, 60000);
console.log("[keep-alive] Started. Interval ID: " + _keepAliveInterval);
'''))
print('Keep-alive started (60s interval)')

# GPU check
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → T4 GPU')
gpu_name = torch.cuda.get_device_name(0)
gpu_cap = torch.cuda.get_device_capability(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu_name} | {gpu_mem:.1f} GB | sm_{gpu_cap[0]}{gpu_cap[1]}')
IS_PRE_AMPERE = gpu_cap[0] < 8
if IS_PRE_AMPERE:
    print('Pre-Ampere → will auto-patch Flash Attention + bf16')
else:
    print('Ampere+ → native support')

In [ ]:
# ====== Cell 2: Mount Google Drive (persistent storage) ======
from google.colab import drive
import os, shutil
from pathlib import Path

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/autoresearch_results')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Results will be saved to: {DRIVE_DIR}')

In [ ]:
# ====== Cell 3: Install uv + Clone + Dependencies ======
import os, subprocess

# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

# Clone autoresearch (skip if already exists)
if not os.path.exists('/content/autoresearch'):
    !git clone https://github.com/karpathy/autoresearch.git /content/autoresearch
else:
    print('autoresearch/ already exists, skipping clone')

os.chdir('/content/autoresearch')
print(f'Working dir: {os.getcwd()}')

# Install deps
!uv sync
!pip install requests pandas -q

In [ ]:
# ====== Cell 4: T4 Patch (auto-detect, skip if Ampere+) ======
import re, torch
from pathlib import Path

train_path = Path('train.py')
if not train_path.exists():
    train_path = Path('autoresearch/train.py')

cap = torch.cuda.get_device_capability(0)
if cap[0] < 8 and train_path.exists():
    print(f'[t4_patch] Pre-Ampere GPU (sm_{cap[0]}{cap[1]}) — patching...')
    text = train_path.read_text()
    original = text

    # Patch 1: Flash Attention 3 → SDPA fallback
    fa3_replacement = '''import torch
_gpu_cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
_is_ampere_plus = _gpu_cap[0] >= 8

if _is_ampere_plus:
    import kernels
    fa3 = kernels.get_kernel("flash_attn3")
else:
    fa3 = None'''

    text = re.sub(
        r'import kernels\s*\nfa3\s*=\s*kernels\.get_kernel\(["\']flash_attn3["\']\)',
        fa3_replacement, text
    )

    # Patch 2: fa3 call → SDPA fallback
    text = re.sub(
        r'y = fa3\.flash_attn_func\(q, k, v, causal=True, window_size=window_size\)',
        '''if fa3 is not None:
            y = fa3.flash_attn_func(q, k, v, causal=True, window_size=window_size)
        else:
            y = torch.nn.functional.scaled_dot_product_attention(
                q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),
                is_causal=True,
            ).transpose(1, 2)''', text
    )

    # Patch 3: bfloat16 → float16
    text = text.replace('torch.bfloat16', 'torch.float16')
    text = re.sub(
        r"autocast\(device_type=['\"]cuda['\"],\s*dtype=torch\.bfloat16\)",
        "autocast(device_type='cuda', dtype=torch.float16)", text
    )

    if text != original:
        train_path.write_text(text)
        print('[t4_patch] Done: Flash Attn 3 → SDPA, bf16 → fp16')
    else:
        print('[t4_patch] Already patched')
elif cap[0] >= 8:
    print(f'[t4_patch] Ampere+ GPU — no patch needed')
else:
    print('[t4_patch] train.py not found')

In [ ]:
# ====== Cell 5: Prepare Data ======
!uv run prepare.py --num-shards 10

In [ ]:
# ====== Cell 6: Robust AutoResearch Runner ======
# Features: OOM recovery, Drive checkpoint, resume support, keep-alive
import argparse, json, os, re, subprocess, sys, time, shutil, gc
from datetime import datetime
from pathlib import Path
import torch

# ── Config ──
ROUNDS = 5  # @param {type:"integer"}
DRIVE_DIR = Path('/content/drive/MyDrive/autoresearch_results')
RESULTS_TSV = DRIVE_DIR / 'results.tsv'
STATE_JSON = DRIVE_DIR / 'runner_state.json'
TRAIN_PY_BACKUP = DRIVE_DIR / 'train.py.backup'

HYPOTHESES = [
    {'name': 'deeper-model',  'desc': 'n_layer 8->12',        'patch': {'target': 'n_layer',          'old': '8',    'new': '12'}},
    {'name': 'wider-model',   'desc': 'n_embd 512->768',      'patch': {'target': 'n_embd',           'old': '512',  'new': '768'}},
    {'name': 'more-heads',    'desc': 'n_head 4->8',          'patch': {'target': 'n_head',           'old': '4',    'new': '8'}},
    {'name': 'larger-batch',  'desc': 'TOTAL_BATCH_SIZE x2',  'patch': {'target': 'TOTAL_BATCH_SIZE', 'mul': 2}},
    {'name': 'higher-lr',    'desc': 'learning_rate x1.5',    'patch': {'target': 'learning_rate',    'mul': 1.5}},
    {'name': 'lower-lr',     'desc': 'learning_rate x0.5',    'patch': {'target': 'learning_rate',    'mul': 0.5}},
    {'name': 'longer-warmup','desc': 'warmup_steps x2',       'patch': {'target': 'warmup_steps',     'mul': 2}},
    {'name': 'window-SSSS',  'desc': 'window all sliding',    'patch': {'target': 'window_pattern',   'old': "'SSSL'", 'new': "'SSSS'"}},
    {'name': 'window-SLSL',  'desc': 'window alternating',    'patch': {'target': 'window_pattern',   'old': "'SSSL'", 'new': "'SLSL'"}},
    {'name': 'seq-4096',     'desc': 'seq_len 2048->4096',    'patch': {'target': 'sequence_len',     'old': '2048', 'new': '4096'}},
]

# ── Helpers ──
def save_state(state):
    STATE_JSON.write_text(json.dumps(state, indent=2))

def load_state():
    if STATE_JSON.exists():
        return json.loads(STATE_JSON.read_text())
    return None

def append_tsv(row):
    header = 'name\tval_bpb\ttrain_loss\tsteps\tduration_s\tkept\tdescription\ttimestamp\n'
    if not RESULTS_TSV.exists():
        RESULTS_TSV.write_text(header)
    with open(RESULTS_TSV, 'a') as f:
        f.write(f"{row['name']}\t{row.get('val_bpb','')}\t{row.get('train_loss','')}\t"
                f"{row.get('steps','')}\t{row.get('duration_s','')}\t{row.get('kept','')}\t"
                f"{row.get('desc','')}\t{row.get('ts','')}\n")

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def run_training(max_retries=2):
    """Run train.py with OOM retry (halve batch on OOM)."""
    tp = Path('train.py')
    for attempt in range(max_retries + 1):
        clear_gpu()
        print(f'  Training (attempt {attempt+1})...', flush=True)
        start = time.time()
        try:
            result = subprocess.run(
                ['uv', 'run', 'train.py'],
                capture_output=True, text=True, timeout=900
            )
            log = result.stdout + result.stderr
        except subprocess.TimeoutExpired:
            return {'error': 'timeout', 'duration_s': round(time.time()-start, 1)}

        duration = round(time.time() - start, 1)

        # Check for OOM
        if 'CUDA out of memory' in log or 'OutOfMemoryError' in log:
            print(f'  OOM detected! Halving batch size...', flush=True)
            text = tp.read_text()
            m = re.search(r'(TOTAL_BATCH_SIZE\s*=\s*)(\d+)', text)
            if m:
                new_bs = max(int(m.group(2)) // 2, 1)
                text = text[:m.start()] + f'{m.group(1)}{new_bs}' + text[m.end():]
                tp.write_text(text)
                print(f'  Batch size → {new_bs}', flush=True)
                clear_gpu()
                continue
            return {'error': 'OOM (cannot reduce further)', 'duration_s': duration}

        # Parse results
        bpb = re.findall(r'val(?:_|\s)bpb[:\s]+([\d.]+)', log)
        loss = re.findall(r'train(?:_|\s)loss[:\s]+([\d.]+)', log)
        steps = re.findall(r'step[:\s]+(\d+)', log)

        val_bpb = float(bpb[-1]) if bpb else None
        if val_bpb is None and result.returncode != 0:
            # Print last 20 lines of log for debugging
            print('  FAILED. Last output:', flush=True)
            for line in log.strip().split('\n')[-20:]:
                print(f'    {line}')
            return {'error': 'training_failed', 'duration_s': duration}

        return {
            'val_bpb': val_bpb,
            'train_loss': float(loss[-1]) if loss else None,
            'steps': int(steps[-1]) if steps else None,
            'duration_s': duration
        }

    return {'error': 'max_retries_exceeded'}

def apply_patch(tp, patch):
    text = tp.read_text()
    target = patch['target']
    if 'old' in patch and 'new' in patch:
        if patch['old'] not in text:
            return False
        text = text.replace(patch['old'], patch['new'], 1)
    elif 'mul' in patch:
        m = re.search(rf'({target}\s*=\s*)([\d.]+)', text)
        if not m:
            return False
        nv = float(m.group(2)) * patch['mul']
        if nv == int(nv):
            nv = int(nv)
        text = text[:m.start()] + f'{m.group(1)}{nv}' + text[m.end():]
    else:
        return False
    tp.write_text(text)
    return True

# ── Main Runner ──
tp = Path('train.py')
if not tp.exists():
    raise FileNotFoundError('train.py not found. Re-run Cell 3.')

# Init git for commit/revert
subprocess.run(['git', 'init', '.'], capture_output=True)
subprocess.run(['git', 'add', '-A'], capture_output=True)
subprocess.run(['git', 'commit', '-m', 'initial', '--allow-empty'], capture_output=True)

# Backup original train.py
if not TRAIN_PY_BACKUP.exists():
    shutil.copy2(tp, TRAIN_PY_BACKUP)

# Check for resume
state = load_state()
if state and state.get('completed_rounds'):
    completed = state['completed_rounds']
    best = state['best_bpb']
    start_round = len(completed)
    print(f'RESUMING from round {start_round} (best so far: {best})')
    print(f'Completed: {[c["name"] for c in completed]}')
    # Restore original train.py for clean patching
    if TRAIN_PY_BACKUP.exists():
        shutil.copy2(TRAIN_PY_BACKUP, tp)
        subprocess.run(['git', 'add', '-A'], capture_output=True)
        subprocess.run(['git', 'commit', '-m', 'restore for resume', '--allow-empty'], capture_output=True)
    # Re-apply kept patches
    for c in completed:
        if c.get('kept'):
            h = next((h for h in HYPOTHESES if h['name'] == c['name']), None)
            if h:
                apply_patch(tp, h['patch'])
                subprocess.run(['git', 'add', '-A'], capture_output=True)
                subprocess.run(['git', 'commit', '-m', f"re-apply kept: {h['name']}"], capture_output=True)
else:
    completed = []
    start_round = 0
    best = None

# Baseline (if not done yet)
if start_round == 0:
    print(f'\n{"="*50}')
    print(f'Round 0/{ROUNDS}: BASELINE')
    print(f'{"="*50}')
    bl = run_training()
    if 'error' in bl:
        print(f'Baseline FAILED: {bl["error"]}')
        raise RuntimeError(f'Baseline failed: {bl}')
    best = bl['val_bpb']
    print(f'  Baseline val_bpb: {best}')
    row = {'name': 'baseline', 'desc': 'Initial baseline', 'kept': True,
           'ts': datetime.utcnow().isoformat(), **bl}
    append_tsv(row)
    completed.append({'name': 'baseline', 'kept': True, 'val_bpb': best})
    save_state({'best_bpb': best, 'completed_rounds': completed})
    print(f'  Saved to Google Drive')

# Experiment rounds
for i, h in enumerate(HYPOTHESES[:ROUNDS], 1):
    # Skip already completed
    if any(c['name'] == h['name'] for c in completed):
        print(f'\nRound {i}/{ROUNDS}: {h["name"]} — ALREADY DONE, skipping')
        continue

    print(f'\n{"="*50}')
    print(f'Round {i}/{ROUNDS}: {h["name"]} — {h["desc"]}')
    print(f'{"="*50}')

    if not apply_patch(tp, h['patch']):
        print('  SKIP (patch target not found)')
        completed.append({'name': h['name'], 'kept': False, 'val_bpb': None, 'skipped': True})
        save_state({'best_bpb': best, 'completed_rounds': completed})
        continue

    subprocess.run(['git', 'add', '-A'], capture_output=True)
    subprocess.run(['git', 'commit', '-m', f"hypothesis: {h['desc']}"], capture_output=True)

    r = run_training()

    if 'error' in r:
        print(f'  FAILED: {r["error"]}')
        kept = False
        subprocess.run(['git', 'reset', '--hard', 'HEAD~1'], capture_output=True)
    elif r['val_bpb'] is not None and r['val_bpb'] < best:
        print(f'  KEPT! val_bpb: {r["val_bpb"]} (was {best})')
        best = r['val_bpb']
        kept = True
    else:
        print(f'  REVERTED val_bpb: {r.get("val_bpb")} (best: {best})')
        subprocess.run(['git', 'reset', '--hard', 'HEAD~1'], capture_output=True)
        kept = False

    row = {'name': h['name'], 'desc': h['desc'], 'kept': kept,
           'ts': datetime.utcnow().isoformat(),
           **{k: v for k, v in r.items() if k != 'error'}}
    append_tsv(row)
    completed.append({'name': h['name'], 'kept': kept, 'val_bpb': r.get('val_bpb')})
    save_state({'best_bpb': best, 'completed_rounds': completed})
    print(f'  Saved to Google Drive')

print(f'\n{"="*50}')
print(f'ALL DONE! Best val_bpb: {best}')
print(f'Results: {RESULTS_TSV}')
print(f'{"="*50}')

In [ ]:
# ====== Cell 7: Show Results ======
import pandas as pd
from pathlib import Path

tsv = Path('/content/drive/MyDrive/autoresearch_results/results.tsv')
if tsv.exists():
    df = pd.read_csv(tsv, sep='\t')
    print(df.to_string(index=False))
    valid = df.dropna(subset=['val_bpb'])
    if len(valid) > 0:
        best_row = valid.loc[valid['val_bpb'].idxmin()]
        print(f"\nBest: {best_row['name']} — val_bpb: {best_row['val_bpb']}")
else:
    print('No results yet. Run Cell 6 first.')

## After Disconnect

If Colab disconnects mid-run:
1. **Reconnect** (Runtime → Reconnect)
2. **Run All** again (Ctrl+F9)
3. The runner auto-detects previous progress from Google Drive and **resumes from the last completed round**

Your results are always safe in `Google Drive/autoresearch_results/results.tsv`